# Task 3 — MSA & Semiglobal Alignment
Prepare sequences for an online MSA, analyze conserved motifs, and prototype a semiglobal alignment variant.

## 1. Initialize Project Environment
Import libraries for FASTA handling, motif scoring, and semiglobal alignment sanity checks.

In [16]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from Bio import AlignIO, SeqIO, pairwise2
from Bio.SeqRecord import SeqRecord

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


def locate_data_root() -> Path:
    here = Path().resolve()
    for base in [here, *here.parents]:
        candidate = base / "data/work/AndreiCod/lab01"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/work/AndreiCod/lab01 relative to current directory"
    )


DATA_ROOT = locate_data_root()
FASTAS = list(DATA_ROOT.glob("*.fa*"))
logging.info("FASTA sources: %s", FASTAS)
assert FASTAS, "Need Lab 1 sequences for MSA prep."


[INFO] FASTA sources: [PosixPath('/workspaces/BDHB-lab/data/work/AndreiCod/lab01/my_tp53.fa'), PosixPath('/workspaces/BDHB-lab/data/work/AndreiCod/lab01/nm000546.fa')]


## 2. Define Configuration Parameters
Centralize which sequences feed the MSA, how many to include, and semiglobal scoring options.

In [17]:
@dataclass
class MSAConfig:
    fasta_path: Path
    max_sequences: int = 5
    export_dir: Path = Path("artifacts")
    semiglobal_settings: Dict[str, float] = None
    msa_trim_bp: Optional[int] = 2000
    semiglobal_trim_bp: Optional[int] = 2000

    def __post_init__(self):
        if self.semiglobal_settings is None:
            self.semiglobal_settings = {
                "match": 2,
                "mismatch": -1,
                "gap_open": -2,
                "gap_extend": 0,
                "penalize_end_gaps": (False, False),
            }

    def describe(self):
        info = asdict(self)
        info["fasta_path"] = str(info["fasta_path"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = MSAConfig(fasta_path=DATA_ROOT / "my_tp53.fa")
CONFIG.describe()


{'fasta_path': '/workspaces/BDHB-lab/data/work/AndreiCod/lab01/my_tp53.fa',
 'max_sequences': 5,
 'export_dir': 'artifacts',
 'semiglobal_settings': {'match': 2,
  'mismatch': -1,
  'gap_open': -2,
  'gap_extend': 0,
  'penalize_end_gaps': (False, False)},
 'msa_trim_bp': 2000,
 'semiglobal_trim_bp': 2000}

In [18]:
def trim_record(record: SeqRecord, max_bp: Optional[int]) -> SeqRecord:
    if max_bp is None or len(record.seq) <= max_bp:
        return record
    trimmed = record[:max_bp]
    trimmed.description = f"{record.description} [trimmed_to_{max_bp}]"
    return trimmed


def trim_records(records: List[SeqRecord], max_bp: Optional[int]) -> List[SeqRecord]:
    return [trim_record(rec, max_bp) for rec in records]


In [19]:
def choose_sequences(cfg: MSAConfig) -> List[SeqIO.SeqRecord]:
    selected: List[SeqIO.SeqRecord] = []
    for record in SeqIO.parse(cfg.fasta_path, "fasta"):
        selected.append(record)
        if len(selected) >= cfg.max_sequences:
            break
    if len(selected) < 3:
        raise ValueError("Need at least 3 sequences for Task 3")
    return selected


selected_records = choose_sequences(CONFIG)
len(selected_records)


3

In [20]:
msa_records = trim_records(selected_records, CONFIG.msa_trim_bp)
semiglobal_inputs = trim_records(selected_records[:2], CONFIG.semiglobal_trim_bp)
logging.info(
    "Prepared %d records for MSA (<=%s bp) and %d for semiglobal alignment",
    len(msa_records),
    CONFIG.msa_trim_bp or "all",
    len(semiglobal_inputs),
)


[INFO] Prepared 3 records for MSA (<=2000 bp) and 2 for semiglobal alignment


## 3. Implement Core Functionality
Export curated FASTA files for Clustal Omega, parse returned alignments, mark conserved motifs, and run a semiglobal demo using Biopython.

In [21]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
subset_path = EXPORT_DIR / "task3_subset_for_msa.fasta"
SeqIO.write(msa_records, subset_path, "fasta")
subset_path


PosixPath('artifacts/task3_subset_for_msa.fasta')

In [22]:
msa_result_path = EXPORT_DIR / "task3_msa_result.clustal"
if msa_result_path.exists():
    with open(msa_result_path) as fh:
        msa_text = fh.read()
    print(msa_text[:2000])
else:
    logging.warning(
        "Upload %s to Clustal Omega, save the output here, and rerun this cell.",
        subset_path,
    )


[WARNING] Upload artifacts/task3_subset_for_msa.fasta to Clustal Omega, save the output here, and rerun this cell.


In [23]:
def load_alignment(path: Path):
    alignment = AlignIO.read(path, "clustal")
    logging.info(
        "Loaded alignment with %d sequences, length %d",
        len(alignment),
        alignment.get_alignment_length(),
    )
    return alignment


if msa_result_path.exists():
    alignment = load_alignment(msa_result_path)
    alignment

In [24]:
def conservation_scores(alignment, threshold: float = 0.9) -> pd.DataFrame:
    columns = []
    aln_len = alignment.get_alignment_length()
    for pos in range(aln_len):
        column = alignment[:, pos]
        counts = pd.Series(list(column)).value_counts()
        top_base = counts.index[0]
        score = counts.iloc[0] / counts.sum()
        columns.append({"position": pos, "top_base": top_base, "score": score})
    df = pd.DataFrame(columns)
    df["conserved"] = df["score"] >= threshold
    return df


if "alignment" in locals():
    conservation_df = conservation_scores(alignment)
    conservation_df.head()

In [25]:
if "alignment" in locals():
    conserved_blocks = conservation_df[conservation_df["conserved"]]
    if conserved_blocks.empty:
        logging.warning(
            "No conserved positions reached the threshold; consider lowering it."
        )
    else:
        conserved_blocks.head()


In [26]:
if (
    "alignment" in locals()
    and "conserved_blocks" in locals()
    and not conserved_blocks.empty
):
    conserved_excerpt = alignment[
        :,
        conserved_blocks.iloc[0]["position"] : conserved_blocks.iloc[0]["position"]
        + 20,
    ]
    print(conserved_excerpt.format("fasta"))


In [27]:
def run_semiglobal(seq1, seq2, cfg: MSAConfig):
    params = cfg.semiglobal_settings
    alignment = pairwise2.align.globalms(
        seq1.seq,
        seq2.seq,
        params["match"],
        params["mismatch"],
        params["gap_open"],
        params["gap_extend"],
        penalize_end_gaps=params["penalize_end_gaps"],
        one_alignment_only=True,
    )
    return alignment[0]


if len(semiglobal_inputs) < 2:
    raise ValueError("Need at least two sequences for semiglobal alignment")

semiglobal_alignment = run_semiglobal(
    semiglobal_inputs[0], semiglobal_inputs[1], CONFIG
)
print(semiglobal_alignment[2])
print(semiglobal_alignment[0][:120])
print(semiglobal_alignment[1][:120])


1073.0
-CT---CCTTGGTTCAA---GTAATTCT--------CCTGCC-TCAGACT---CCAGAGTA--GCTGGGA---TTAC----AGGCGCCC--GCC---ACC---ACGC---CCAGCTAATT
CCTAACCCT------AACCCATAACCCTAACCCTAACCTACCCTAACCCTAACCC----TAACCCT---AACCTAACCCTAA----CCCTAACCCTAACCCTAACCCTAACC--CTAA--


## 4. Validate with Unit Tests
Sanity-check the conservation scoring and semiglobal helper on synthetic data to avoid surprises.

In [28]:
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord


def test_conservation_scores():
    dummy_alignment = MultipleSeqAlignment(
        [
            SeqRecord(Seq("AAAA"), id="s1"),
            SeqRecord(Seq("AAAT"), id="s2"),
            SeqRecord(Seq("AAAC"), id="s3"),
        ]
    )
    df = conservation_scores(dummy_alignment, threshold=0.66)
    assert df.loc[df["position"] == 0, "conserved"].iloc[0]


def test_semiglobal_alignment():
    seq1 = SeqRecord(Seq("TTTAAA"), id="a")
    seq2 = SeqRecord(Seq("AAA"), id="b")
    aln = run_semiglobal(seq1, seq2, CONFIG)
    assert "AAA" in aln[0]


if "alignment" not in locals():
    test_conservation_scores()
    logging.info("Conservation scoring tests passed on dummy data.")

test_semiglobal_alignment()
logging.info("Semiglobal helper test passed.")

[INFO] Conservation scoring tests passed on dummy data.
[INFO] Semiglobal helper test passed.


## 5. Analyze Performance Metrics
Record runtime of the conservation computation and semiglobal alignment to estimate scaling behavior.

In [29]:
import time

perf_rows = []
if "alignment" in locals():
    start = time.perf_counter()
    conservation_scores(alignment)
    perf_rows.append({"task": "conservation", "runtime_s": time.perf_counter() - start})

if len(semiglobal_inputs) >= 2:
    start = time.perf_counter()
    run_semiglobal(semiglobal_inputs[0], semiglobal_inputs[1], CONFIG)
    perf_rows.append({"task": "semiglobal", "runtime_s": time.perf_counter() - start})
else:
    logging.warning("Skipping semiglobal benchmark; not enough trimmed sequences.")

pd.DataFrame(perf_rows)


,task,runtime_s
0,semiglobal,0.523709


In [30]:
if (
    "alignment" in locals()
    and "conserved_blocks" in locals()
    and not conserved_blocks.empty
):
    conserved_blocks.to_csv(EXPORT_DIR / "task3_conserved_blocks.csv", index=False)
    if "conserved_excerpt" in locals():
        with open(EXPORT_DIR / "task3_conserved_excerpt.fasta", "w") as fh:
            fh.write(conserved_excerpt.format("fasta"))

with open(EXPORT_DIR / "task3_semiglobal_alignment.txt", "w") as fh:
    seq1, seq2, score, start, end = semiglobal_alignment
    fh.write(f"Score: {score}\nStart:{start} End:{end}\n")
    fh.write(seq1 + "\n" + seq2 + "\n")

print("Artifacts saved to", EXPORT_DIR)


Artifacts saved to artifacts
